# 05 — КБЛ: полный пайплайн видео → игроки → метры → демо

Сквозной прогон на `kz_clip01` (Национальная лига Казахстана, статичная камера,
32 с): детекция игроков → **авто-ROI из гомографии** → трекинг (ByteTrack) →
координаты в метрах → траектории, дистанции, скорости и **видео
«трансляция + миникарта»**.

Опирается на ручную калибровку `points_kz_clip01.json` (ошибка репроекции
0.07 м, кадр 480). Камера внутри клипа статична — одна матрица на весь клип.

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import sys, time, json, warnings, subprocess
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)  # sv.ByteTrack deprecated в 0.29
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
import supervision as sv
from ultralytics import YOLO

sys.path.insert(0, str(Path("..").resolve()))
from src.court import draw_court, court_polylines
from src.config import (DATA_PROCESSED_DIR, OUTPUT_DIR, MODELS_DIR, PROJECT_ROOT,
    DETECT_MODEL, DETECT_IMGSZ, DETECT_DEVICE, PERSON_CLASS_ID, TRACK_DET_FLOOR,
    TRACK_ACTIVATION_THRESHOLD, TRACK_LOST_BUFFER, TRACK_MIN_MATCHING_THRESHOLD, TRACK_FRAME_RATE)

CLIP_NAME = "kz_clip01"
CLIP = DATA_PROCESSED_DIR / f"{CLIP_NAME}.mp4"
DEVICE = DETECT_DEVICE
print(f"клип {CLIP.name}; детекция {DETECT_MODEL}@{DETECT_IMGSZ}; трекер act={TRACK_ACTIVATION_THRESHOLD}")

## Гомография из ручной калибровки

$H$ из размеченных точек (правая половина). `img_to_court` переводит пиксель
кадра в метры ($X$ 0..28, $Y$ 0..15).

In [ ]:
spec = json.loads((PROJECT_ROOT / "points_kz_clip01.json").read_text())
img_pts = np.array([p["image"] for p in spec["points"]], float)
wld_pts = np.array([p["world"] for p in spec["points"]], float)
H, _ = cv2.findHomography(wld_pts, img_pts, cv2.RANSAC, 10)   # world -> image
Hi = np.linalg.inv(H)                                          # image -> world

def img_to_court(uv):
    uv = np.atleast_2d(uv).astype(float)
    p = np.hstack([uv, np.ones((len(uv), 1))]); q = (Hi @ p.T).T
    return q[:, :2] / q[:, 2:3]

err = np.linalg.norm(img_to_court(img_pts) - wld_pts, axis=1)
print(f"точек калибровки: {len(spec['points'])}; ошибка репроекции: средняя {err.mean():.2f} м, макс {err.max():.2f} м")

## Сквозной прогон: детекция → авто-ROI → трекинг → метры

**Авто-ROI:** детекция принимается, только если её foot-точка проецируется
внутрь прямоугольника площадки (28×15 с запасом 1 м) — гомография сама отсекает
скамейку/судей.

In [ ]:
model = YOLO(str(MODELS_DIR / DETECT_MODEL))
tracker = sv.ByteTrack(track_activation_threshold=TRACK_ACTIVATION_THRESHOLD,
    lost_track_buffer=TRACK_LOST_BUFFER, minimum_matching_threshold=TRACK_MIN_MATCHING_THRESHOLD,
    frame_rate=TRACK_FRAME_RATE)

rows = []; cap = cv2.VideoCapture(str(CLIP)); fi = 0; t0 = time.perf_counter()
while True:
    ok, fr = cap.read()
    if not ok: break
    res = model.predict(fr, device=DEVICE, imgsz=DETECT_IMGSZ, conf=TRACK_DET_FLOOR, verbose=False)[0]
    det = sv.Detections.from_ultralytics(res); det = det[det.class_id == PERSON_CLASS_ID]
    if len(det):
        xy = det.xyxy; foot = np.column_stack([(xy[:, 0] + xy[:, 2]) / 2, xy[:, 3]])
        c = img_to_court(foot)
        det = det[(c[:, 0] >= -1) & (c[:, 0] <= 29) & (c[:, 1] >= -1) & (c[:, 1] <= 16)]  # авто-ROI
    det = tracker.update_with_detections(det)
    for (x1, y1, x2, y2), conf, tid in zip(det.xyxy, det.confidence, det.tracker_id):
        fx, fy = (x1 + x2) / 2, y2; cx, cy = img_to_court([fx, fy])[0]
        rows.append((fi, int(tid), float(conf), float(fx), float(fy), float(cx), float(cy)))
    fi += 1
cap.release()
dt = time.perf_counter() - t0
df = pd.DataFrame(rows, columns=["frame", "track_id", "conf", "foot_x", "foot_y", "court_x", "court_y"])
print(f"кадров {fi} за {dt:.1f}с ({fi/dt:.0f} FPS); строк {len(df)}")
print(f"уникальных треков: {df.track_id.nunique()}; медиана игроков в кадре: {int(df.groupby('frame').size().median())}")

## Сглаживание координат

Foot-точка дрожит, а редкие подмены ID дают «телепорты». Для чистых метрик и
следов на каждый трек — скользящая **медиана окном 7** (устойчива к выбросам).
Столбцы `court_x_s`, `court_y_s`. Сохраняем parquet со сглаженными координатами.

In [ ]:
df = df.sort_values(["track_id", "frame"]).reset_index(drop=True)
df["court_x_s"] = df.groupby("track_id")["court_x"].transform(lambda s: s.rolling(7, center=True, min_periods=1).median())
df["court_y_s"] = df.groupby("track_id")["court_y"].transform(lambda s: s.rolling(7, center=True, min_periods=1).median())
OUT = OUTPUT_DIR / "tracks_kz_clip01_court.parquet"; df.to_parquet(OUT, index=False)
print(f"сохранено {OUT.name} со сглаженными координатами")
df.head(3)

## Валидация: игроки одного кадра на схеме в метрах

In [ ]:
F = 480
cap = cv2.VideoCapture(str(CLIP)); cap.set(cv2.CAP_PROP_POS_FRAMES, F); _, frame = cap.read(); cap.release()
d = df[df.frame == F]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(22, 7))
a1.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
a1.scatter(d.foot_x, d.foot_y, c="lime", s=60, ec="black", zorder=5)
a1.set_title(f"Кадр {F}: {len(d)} игроков на площадке (авто-ROI)"); a1.axis("off")
draw_court(a2); a2.axvspan(20, 28, alpha=0.07, color="green")
a2.scatter(d.court_x, d.court_y, c="tab:red", s=130, ec="black", zorder=5)
a2.set_title("Те же игроки в метрах на схеме ФИБА")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "05_kz_frame_court.png", dpi=95, bbox_inches="tight"); plt.show()

## Траектории и метрики (дистанция, скорость) в метрах

По сглаженным координатам; шаги >1.5 м/кадр (остаточные телепорты) отсечены.

In [ ]:
def stats(t):
    t = t.sort_values("frame"); cx = t.court_x_s.values; cy = t.court_y_s.values; f = t.frame.values
    dist = 0.0; sp = []
    for i in range(1, len(f)):
        if f[i] - f[i - 1] == 1:
            step = np.hypot(cx[i] - cx[i - 1], cy[i] - cy[i - 1])
            if step < 1.5:
                dist += step; sp.append(step * 30)
    return pd.Series({"кадров": len(f), "сек": round(len(f) / 30, 1), "дистанция_м": round(dist, 1),
                      "ср_скорость_мс": round(np.mean(sp), 2) if sp else 0})

g = df.groupby("track_id").apply(stats).sort_values("кадров", ascending=False)
top = g.head(6)
print(top.to_string())

fig, ax = plt.subplots(figsize=(13, 7.5)); draw_court(ax); ax.axvspan(20, 28, alpha=0.07, color="green")
for tid, col in zip(top.head(4).index, ["tab:red", "tab:blue", "tab:green", "tab:orange"]):
    t = df[df.track_id == tid].sort_values("frame")
    ax.plot(t.court_x_s, t.court_y_s, "-", color=col, lw=1.5, alpha=0.85, label=f"#{tid}: {g.loc[tid,'дистанция_м']:.0f} м")
ax.legend(loc="upper left", fontsize=9)
ax.set_title("Траектории игроков kz_clip01 в метрах (сглажено; зелёная зона — точная калибровка)")
plt.savefig(OUTPUT_DIR / "05_kz_trajectories.png", dpi=110, bbox_inches="tight"); plt.show()

## Демо-видео: трансляция + миникарта в метрах

Слева кадр с отмеченными игроками (ID + точка ног), справа — вид сверху в метрах
со следами по **сглаженным** координатам. След **разрывается** на скачке >1.5 м
(подмена ID больше не «простреливает» через площадку). Пишем mp4v (cv2), затем
перекодируем в H.264 для совместимости (QuickTime/браузер).

In [ ]:
# Миникарта: метры -> пиксели (высота 720 = 15 м с полями)
Hh, margin = 720, 45
scale = (Hh - 2 * margin) / 15.0
Wm = int(2 * margin + 28 * scale)
def c2p(cx, cy):
    return int(margin + cx * scale), int(Hh - (margin + cy * scale))

base = np.full((Hh, Wm, 3), 245, np.uint8)
cv2.rectangle(base, c2p(20, 0), c2p(28, 15), (225, 245, 225), -1)  # зона точной калибровки
for poly in court_polylines():
    cv2.polylines(base, [np.array([c2p(x, y) for x, y in poly], np.int32)], False, (60, 60, 60), 2, cv2.LINE_AA)
cv2.putText(base, "METERS (top-down)", (margin, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

PAL = [(66,66,244),(244,133,66),(66,244,66),(244,66,220),(66,220,244),(180,66,244),(66,166,255),
       (0,200,160),(200,120,40),(120,200,80),(40,120,220),(200,60,120),(0,160,255),(160,80,200),
       (80,200,200),(220,180,40),(120,120,255),(60,180,120)]
def col(tid): return PAL[tid % len(PAL)]

by_frame = {fi: sub for fi, sub in df.groupby("frame")}
RAW = OUTPUT_DIR / "kz_side_by_side.mp4"
writer = cv2.VideoWriter(str(RAW), cv2.VideoWriter_fourcc(*"mp4v"), 30, (1280 + Wm, 720))
trails, prev = {}, {}
cap = cv2.VideoCapture(str(CLIP)); fi = 0
while True:
    ok, frame = cap.read()
    if not ok: break
    mini = base.copy(); active = set()
    sub = by_frame.get(fi)
    if sub is not None:
        for r in sub.itertuples():
            c = col(r.track_id); active.add(r.track_id)
            fx, fy = int(r.foot_x), int(r.foot_y)
            cv2.circle(frame, (fx, fy), 6, c, -1, cv2.LINE_AA); cv2.circle(frame, (fx, fy), 6, (0, 0, 0), 1)
            cv2.putText(frame, str(r.track_id), (fx - 8, fy - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, c, 2)
            if r.track_id in prev and np.hypot(r.court_x_s - prev[r.track_id][0], r.court_y_s - prev[r.track_id][1]) > 1.5:
                trails[r.track_id] = []  # разрыв следа на телепорте/подмене
            prev[r.track_id] = (r.court_x_s, r.court_y_s)
            trails.setdefault(r.track_id, []).append(c2p(r.court_x_s, r.court_y_s))
            trails[r.track_id] = trails[r.track_id][-30:]
    for tid, tr in trails.items():
        if len(tr) > 1: cv2.polylines(mini, [np.array(tr, np.int32)], False, col(tid), 2, cv2.LINE_AA)
        if tid in active:
            cv2.circle(mini, tr[-1], 9, col(tid), -1, cv2.LINE_AA); cv2.circle(mini, tr[-1], 9, (0, 0, 0), 1)
            cv2.putText(mini, str(tid), (tr[-1][0] + 8, tr[-1][1]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    cv2.putText(frame, "kz_clip01", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    writer.write(cv2.hconcat([frame, mini])); fi += 1
cap.release(); writer.release()

FINAL = OUTPUT_DIR / "kz_demo.mp4"
try:
    subprocess.run(["ffmpeg", "-y", "-i", str(RAW), "-c:v", "libx264", "-pix_fmt", "yuv420p",
                    "-crf", "23", str(FINAL)], check=True, capture_output=True)
    print(f"видео (H.264): {FINAL}  |  {fi} кадров, {1280+Wm}x720")
except Exception as e:
    print(f"ffmpeg недоступен — остаётся mp4v: {RAW}")

## Итог

Одна ручная разметка (7 точек) → сквозной пайплайн: видео → игроки → метры →
траектории, скорости и демо-видео «трансляция + тактическая миникарта».
Медиана 10 игроков в кадре (авто-ROI из гомографии), координаты сглажены.
Точнее всего правая половина (зона калибровки), где идёт игра.

**Ограничения (BACKLOG):** левая половина — экстраполяция; редкие подмены ID
(след разрывается, но ID может смениться). Следующее: этапы 5–6 (тепловые
карты, зоны, команды по цвету формы); при желании — второй клип / дообучение
court-модели под КБЛ для авто-разметки.